# Soul Tower Analytics Notebook

This notebook is your exploration workspace for Soul Tower's game data. Every cell is runnable and annotated, so you can learn the analytics system by walking top to bottom.

**What you can do here:**
- Load every spreadsheet into a DataFrame
- Search for any ability across every sheet at once
- Convert die-based mana costs to numeric bounds
- Group cards by type, subtype, or origin
- Merge static card data with live Flask analytics

**Prerequisites:**
- You've run `python main.py --fresh` at least once to populate `data/cache/`
- If you want live analytics, the Flask server must be running on `localhost:5050`

**How to extend this notebook:**
- New analytics questions become new cells, not new files
- When a cell becomes reusable, move it to `src/analytics/query.py` as a function
- Anything that returns a DataFrame can be chained with pandas methods — `.groupby()`, `.query()`, `.merge()`

## 1. Setup

Make `query.py` importable. The `sys.path.insert` hack lets you run this notebook from anywhere in the project.

In [1]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import src.analytics.query
project_root = Path.cwd()
while project_root.name and not (project_root / 'src' / 'analytics' / 'query.py').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.analytics import query
import pandas as pd

# Show all columns in DataFrames (pandas truncates by default)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

print(f'Project root: {project_root}')
query.summary()

Project root: C:\Users\sethg\Soul Tower\soul_tower_kit
Soul Tower cache summary
  hero                  176 rows   cols: ['ID', 'Name', 'Passive', 'Passive Effect', 'Health']...
  common_cards          705 rows   cols: ['ID', 'Name', 'Origin', 'Type', 'Subtype']...
  legendary             354 rows   cols: ['Origin', 'ID', 'Game Name', 'Name', 'Type']...
  calamity               35 rows   cols: ['ID', 'Name', 'Effect', 'Curse Source', 'Curse Effect']...
  villain                21 rows   cols: ['ID', 'Name', 'Origin', 'Passive', 'Passive Effect']...

Cache dir: C:\Users\sethg\Soul Tower\soul_tower_kit\data\cache


## 2. Load everything

One call gives you a dict of five DataFrames keyed by sheet name. Store them in a variable and use as needed.

In [23]:
sheets = query.load_all()

heroes   = sheets['hero']
commons  = sheets['common_cards']
legends  = sheets['legendary']
calams   = sheets['calamity']
villains = sheets['villain']

print(f'Heroes:         {len(heroes)}')
print(f'Common cards:   {len(commons)}')
print(f'Legendary:      {len(legends)}')
print(f'Calamities:     {len(calams)}')
print(f'Villains:       {len(villains)}')

Heroes:         176
Common cards:   705
Legendary:      354
Calamities:     35
Villains:       21


## 3. Peek at one sheet

`.head()` shows the first few rows. `.columns` shows you what's available for querying.

In [24]:
heroes.head()

,ID,Name,Passive,Passive Effect,Health,Might,Speed,Luck,Arcana,Origin,Flavor Text,Card1 Name,Card1 Type,Card1 Subtype,Card1 Cost,Card1 Effect,Card1 Flavor Text,Card2 Name,Card2 Type,Card2 Subtype,Card2 Cost,Card2 Effect,Card2 Flavor Text,Difficulty,Game Name,Alignment,Display Name,Nickname,Status,Image URL,Synergy,Hints,Icons
0,1,"Akiem, Reaper of Peace",Eyes of Blight,"Whenever You Enchant, Fortune 3\n\nWhenever You Rally, Enemy Weaken Speed 1",3,4,6,3,2,Cresia,"The Nexus's power is split across the three Relics of Humanity. Peace, Wrath...",Demise of the Blight,Ritual,Entropy,7,Deal 5 Hero\nForce Charge 2: Obliterate 1 the Target's Hand\nthe Target Ench...,"We say Blight, but there are a few types. The shades dissapate just fine. Fo...",Message for Peace,Brutal,Normal,3,Attune 2\nRally: Your Hero Boost Arcana 2,The look on her face when we saw the Message of Resolve said it all. I was n...,1,akiem,Blessed,"Akiem, Reaper of Peace",Akiem,Live,Game Images/Hero/akiem.jpg,,,
1,2,"Alverus, Devil's Finger",Devil's Favor,"Whenever You Play Normal Ritual, Blood Magic 15: Fortune 3\n\nWhenever You M...",4,4,3,3,4,Solenak,"A devil always swears its deals in Blood, and makes oaths of Hellfire. Oh Sl...",Infernal Contract,Ritual,Entropy,6,Forbid 1: Set Burn 1\nBlood Price 6: Exhume 12\n\nKill all Heroes,The powers of the Hand's Hell is limitless!,Devil's Immortality,Ritual,Entropy,3,Blood Price 3: Your Hero Set Indestructible 1\nExhume 3,Life for a price.,2,alverus,Cursed,"Alverus, Devil's Finger",Alverus,Live,Game Images/Hero/alverus.jpg,"Kalibar, Jeset, Verrida, Goliath, Lasmeer","Blood Magic, Fortune, Mercy",
2,3,"Anemos, the Insane",Joyous Madness,"Wild +1\n\nWhenever You Fortune #, Heal Fortuned Hero.",4,5,3,4,2,"Lashiq, the Unliving Sands",A man? Not quite so. \nA Devil? Not quite so. \nPositively mad with the Dark...,Enlightened Madness,Ritual,Entropy,4,Wild 6\nFortune 3\nBolster: Heal double Fortuned Hero,Things cost what they cost and do what they do...what's the cost of a little...,Warp Release: Insanity Rises,Ritual,Entropy,10,Fortune 6\nForce Blood Price Fortuned: Enemies Weaken Might Fortuned\nYou Bo...,"For centuries, The Grimoire of Chaos only had three Releases. But with the W...",4,anemos,Blessed,"Anemos, the Insane",Anemos,Live,Game Images/Hero/anemos.jpg,"Jeki, Negaknight, Nth Knight",,
3,4,"Anok'sa, the Buried Empress",Caress of the Tomb,"Whenever You Harness, Attune 2.\n\nWhenever Embalm, ME Set Toughness 1d4.",3,4,2,5,4,The Dark,,Lover's Snare,Spell,Instant,4,Harness 3: Add 2 Deny to Lover's Snare\nDeny 4\nMercy: Ally Set Toughness 1d...,Doing whatever it takes to prove a point... it has its benefits.,Embrace of the Mistress,Brutal,Normal,3,Attune 3\nUpcast 3: Obliterate 1 Your Hand\nBolster: Deal Obliterated Cost Hero,"Before the Sands began choking the Earth, my father would present fighters f...",2,anoksa,Cursed,"Anok'sa, the Buried Empress",Anok'sa,Live,Game Images/Hero/anoksa.jpg,Soul Legion Sorcerer,,
4,5,"Anya, She Who Ends Night",Waking Sky,"Whenever You Attune #, Foe Attune -1d2\n\nWhenever You Exhume #, Deal Exhume...",5,3,3,3,4,Funar'e,"For a thousand of years, the Longnight terrorized all of Funar'e. And now th...",Soul Exchange,Ritual,Normal,3,Pray: Remove 1 Tomb counter;\nRemove all Tomb counters 1 Card Your Tomb\nExh...,To save a soul lost in the Longnight means to condemn another.,Tethered Life,Spell,Normal,7,Harness 5: Remove 1 Tomb counter\nMercy: Exhume 8\n : Bolster: You ...,"Even when you reject My Gifts, I will still come through for you.",4,anya,Blessed,"Anya, She Who Ends Night",Anya,Live,Game Images/Hero/anya.jpg,,,


In [25]:
# See every column name in the Hero sheet
list(heroes.columns)

['ID',
 'Name',
 'Passive',
 'Passive Effect',
 'Health',
 'Might',
 'Speed',
 'Luck',
 'Arcana',
 'Origin',
 'Flavor Text',
 'Card1 Name',
 'Card1 Type',
 'Card1 Subtype',
 'Card1 Cost',
 'Card1 Effect',
 'Card1 Flavor Text',
 'Card2 Name',
 'Card2 Type',
 'Card2 Subtype',
 'Card2 Cost',
 'Card2 Effect',
 'Card2 Flavor Text',
 'Difficulty',
 'Game Name',
 'Alignment',
 'Display Name',
 'Nickname',
 'Status',
 'Image URL',
 'Synergy',
 'Hints',
 'Icons']

## 4. Find every card and hero that uses an ability

`query.find_ability()` is the most powerful exploration tool. It searches every effect and passive column across every sheet for a keyword and returns a unified DataFrame showing exactly where each match was found.

Try changing `'Enchant'` to any ability you care about: `'Attune'`, `'Forsake'`, `'Deal'`, `'Blood Price'`.

In [26]:
query.find_ability('Enchant')

,source,name,type,subtype,origin,cursed,matched_in,snippet
0,hero,Akiem,,,Cresia,,Passive Effect,"Whenever You Enchant, Fortune 3\n\nWhenever You Rally, Enemy Weaken Speed 1"
1,hero,Parasite No. 15,,,The World of Tomorrow,,Passive Effect,"Whenever You Enchant, Deal 1d3 Hero & Heal Dealt ME\n\nWhenever Weaken #, Ob..."
2,hero,Torn,,,Imperius,,Passive Effect,"Your Cards with Equip Cost -5\nInstead if Your Cards with Equip, Enchant."
3,hero,Violetta,,,Imperius,,Passive Effect,"Whenever you Enchant, the Target Weaken 2 Might & Arcana.\n\nYou Obliterate +1"
4,hero,Wex,,,None,,Passive Effect,"Whenever You Fail Wild, Crystallize the Wild Card.\n\nWhenever You Enchant, ..."
...,...,...,...,...,...,...,...,...
105,legendary,Swarm Lord Blessing,Ritual,Entropy,"Verengral, Swarm Host",,Effect,"Attune 3\nthe Target Enchant: Whenever Your Hero Dealt Damage, Your Hero Set..."
106,legendary,Unload,Spell,Instant,"Evin, Priest Shot",,Effect,Enchant: Speed 5\nForsake 1: Your Brutals cost 1 less.
107,legendary,Vivid Colors,Brutal,Reaction-Play,Martial Artist Ko,,Effect,Charge 3: the Target Boost Speed 4\nthe Target Enchant: Increase your die ro...
108,legendary,Watchful Binding,Brutal,Reaction-Enemy Attack,"Kanos, The Mighty",,Effect,Enemy Enchant: Deal Your Hero 2d4 Hero\n : Your Enemies Set To...


In [27]:
# Filter to just one source
enchants = query.find_ability('Enchant')
enchants[enchants['source'] == 'legendary']

,source,name,type,subtype,origin,cursed,matched_in,snippet
72,legendary,Blood Iron Artifact,Ritual,Normal,"Krea, the Red Dwarf",,Effect,Blood Price 3: You Enchant: Reduce your next Brutal Card Play mana cost to 0...
73,legendary,Book Worm,Spell,Instant,"Phastos, the Glim",,Effect,Set Agony 1d4 + Might\nDeal Agony Stack the Target\nthe Target Enchant: You ...
74,legendary,Crystal Wrath,Spell,Instant,"Wex, Crystal Scar",,Effect,Heal 3 Hero\nYou Enchant: Your Crystals have Inspire 1.
75,legendary,Demise of the Blight,Ritual,Entropy,"Akiem, Reaper of Peace",,Effect,Deal 5 Hero\nForce Charge 2: Obliterate 1 the Target's Hand\nthe Target Ench...
76,legendary,Devour Intellect,Brutal,Normal,Parasite No. 15,,Effect,Charge 2: Foe Weaken Arcana 2\nFoe Enchant: You Silence 1
77,legendary,Downdraft,Brutal,Reaction-Attack,"Snuff, the Howler Mage",,Effect,Deny 8\nBolster: Attacker Weaken Speed 1\nAttacker Enchant: Speed -1
78,legendary,Empower Host,Ritual,Entropy,"Ichicha, Blind Demoness",,Effect,Enchant: Power +2\nCharge 4: Heal 3 Enchanted\nRally: Enchanted Draw 1
79,legendary,Explosive Bolts,Brutal,Normal,"Obsolet, Steel Eye",,Effect,Risk 1d4\nBolster: Enchant: Your Power has +1d4
80,legendary,Explosive Shrapnel Belt,Brutal,Normal,Goblin Engineer,,Effect,Equip: Your Hero Set Toughness 1\nEnchant: Deal 4d2 Heroes\n : Y...
81,legendary,Friend of a Friend,Spell,Normal,"Paranorah, Demon Hostess",,Effect,Wild 3\nBolster: All Attune 3\nEnchant: Health -3


In [28]:
# Or group by source to see counts
query.find_ability('Enchant').groupby('source').size()

source
common_cards    63
hero             9
legendary       38
dtype: int64

## 5. Card cost analysis

Card costs in the spreadsheet are strings. Some are plain numbers (`'3'`), some are die expressions (`'2d4'`). `query.add_cost_columns()` parses them all and gives you three numeric columns: `cost_min`, `cost_max`, `cost_avg`.

Once those are in place, all the usual pandas math works.

In [29]:
commons_costed = query.add_cost_columns(commons)
commons_costed[['Name', 'Type', 'Subtype', 'Cost', 'cost_min', 'cost_max', 'cost_avg', 'cost_is_die']].head(10)

,Name,Type,Subtype,Cost,cost_min,cost_max,cost_avg,cost_is_die
0,Cursed Wish,Ritual,Entropy,4,4,4,4.0,False
1,Dark Heart,Ritual,Entropy,6,6,6,6.0,False
2,Dark Release: Solemn Promise,Ritual,Entropy,7,7,7,7.0,False
3,Demon Release: Soul Combustion,Ritual,Entropy,6,6,6,6.0,False
4,Demonic Greed,Ritual,Entropy,6,6,6,6.0,False
5,Fated Demise,Ritual,Entropy,2,2,2,2.0,False
6,Oathsworn Shield,Ritual,Entropy,11,11,11,11.0,False
7,Savage Brand,Ritual,Entropy,4,4,4,4.0,False
8,Sinister Stone,Ritual,Entropy,3,3,3,3.0,False
9,Walk Beside Death,Ritual,Entropy,9,9,9,9.0,False


In [30]:
# Average cost per card type
commons_costed.groupby('Type')['cost_avg'].agg(['mean', 'min', 'max', 'count'])

,mean,min,max,count
Type,,,,
Brutal,4.165939,0.0,12.0,229
Ritual,4.746479,0.0,12.0,284
Spell,5.033854,0.0,12.0,192


In [31]:
# Which cards use die-based costs?
commons_costed[commons_costed['cost_is_die']][['Name', 'Type', 'Subtype', 'Cost', 'cost_min', 'cost_max']]

,Name,Type,Subtype,Cost,cost_min,cost_max
14,Ballistic Hex,Ritual,Entropy,1d8,1,8
15,Brutal Hex,Ritual,Entropy,1d4,1,4
16,Burning Hex,Ritual,Entropy,1d6,1,6
22,Foresee Life,Ritual,Entropy,1d4,1,4
45,Soul Blink,Ritual,Entropy,1d8,1,8
46,Soul Invigoration,Ritual,Entropy,1d8,1,8
60,Book of Blood,Ritual,Entropy,1d10,1,10
61,Book of Scrap,Ritual,Entropy,1d10,1,10
90,Strip Destiny,Ritual,Entropy,6d2,6,12
114,Arctic Plunge,Spell,Instant,1d4,1,4


## 6. All playable cards in one table

Common cards and Legendary cards live in separate sheets but share most columns. `query.cards_by_type()` unions them with a `card_source` column so you can treat them as one dataset.

In [32]:
all_cards = query.cards_by_type()
print(f'Total cards: {len(all_cards)}')

# Count by source and type
all_cards.groupby(['card_source', 'Type']).size().unstack(fill_value=0)

Total cards: 1059


Type,Brutal,Ritual,Spell
card_source,,,
common,229,284,192
legendary,133,119,102


## 7. Group by Origin

For Heroes, this shows stat averages per Origin — useful for checking design rules like the Imanis 5 Health minimum.

For card sheets, it shows card count and cost ranges.

In [33]:
print(heroes[["Health", "Might", "Speed", "Luck", "Arcana"]].dtypes)

Health    object
Might     object
Speed     object
Luck      object
Arcana    object
dtype: object


In [34]:
query

<module 'src.analytics.query' from 'C:\\Users\\sethg\\Soul Tower\\soul_tower_kit\\src\\analytics\\query.py'>

In [35]:
query.by_origin('hero')

Health             Might             Speed              Luck            Arcana        
                                mean min max      mean min max      mean min max      mean min max      mean min max
Origin                                                                                                              
4th Hell                    4.250000   3   6  3.687500   2   5  3.250000   2   6  3.375000   2   6  3.437500   2   6
Cresia                      4.000000   3   6  3.166667   2   4  3.666667   2   6  3.166667   2   4  4.000000   2   7
Funar'e                     4.136364   2   7  3.181818   2   6  3.500000   2   6  3.636364   2   6  3.545455   2   6
Immanis                     5.000000   3   7  3.357143   2   5  3.785714   2   6  3.071429   2   7  2.785714   2   6
Imperius                    4.363636   3   7  3.863636   2   6  3.227273   2   6  3.545455   2   6  3.090909   2   5
Lashiq, the Unliving Sands  4.714286   2   7  3.380952   2   6  2.952381   2   5  3.619048   2   6  3.333333   2   7
Motoro                      4.454545   3   6  4.090909   2   6  3.000000   2   5  3.727273   2   6  2.727273   2   4
Mythic                      5.000000   2   8  4.500000   3   7  4.166667   3   5  4.333333   3   7  4.000000   2   7
None                        3.450000   2   6  3.500000   2   5  3.100000   2   5  3.800000   2   7  4.150000   2   7
Solenak                     3.800000   2   5  3.000000   2   5  3.466667   2   5  4.133333   3   5  3.600000   2   7
The Dark                    3.916667   2   7  3.750000   2   6  2.833333   2   7  3.416667   2   5  4.083333   2   7
The World of Tomorrow       3.454545   2   6  3.363636   2   6  2.818182   2   4  4.181818   2   7  4.181818   2   7

In [36]:
# Check: do all Imanis heroes have Health >= 5?
imannis = heroes[heroes['Origin'] == 'Imannis']
imannis[['Nickname', 'Health', 'Might', 'Speed', 'Luck', 'Arcana']]

,Nickname,Health,Might,Speed,Luck,Arcana


In [37]:
# Flag any violations
violations = heroes[(heroes['Origin'] == 'Imanis') & (heroes['Health'].astype(int) < 5)]
if len(violations) == 0:
    print('✓ All Imanis heroes meet the 5 Health minimum')
else:
    print(f'✗ {len(violations)} violations:')
    print(violations[['Nickname', 'Origin', 'Health']])

✓ All Imanis heroes meet the 5 Health minimum


## 8. Find cards a specific Hero connects to

Every Legendary card has an Origin field pointing to its Hero. This lets you look up Akiem's signature cards without scanning by hand.

In [39]:
legends.head()

,Origin,ID,Game Name,Name,Type,Subtype,Cost,Effect,Flavor Text,Image URL,Card URL,Hero Image
0,Pyrodancer Allia,allia.a_dance_of_fire,a_dance_of_fire,A Dance of Fire,Ritual,Entropy,1d8,Risk 2 + Mana Paid\nCatalyst: Brutal: Heroes Set Burn 1\nCatalyst: Ritual: H...,,Game Images/Hero/allia.jpg,Game Images/hero cards/a_dance_of_fire.jpg,allia.jpg
1,Soul of the Dragon,soul_dragon.advanced_arcane_lance,advanced_arcane_lance,Advanced Arcane Lance,Spell,Instant,6,Deal 1d6 Hero.\nThreshold 4: Repeat 1\nDeal 2 Hero.,"When perfected, the results speak for themselves.",Game Images/Hero/soul_dragon.jpg,Game Images/hero cards/advanced_arcane_lance.jpg,soul_dragon.jpg
2,"Halifor, Aether Magus",halifor.aether_reap,aether_reap,Aether Reap,Spell,Instant,7,Charge 4: Set Agony 5\nUpcast 4: Deal 7 the Target\nDrain 6 the Target,,Game Images/Hero/halifor.jpg,Game Images/hero cards/aether_reap.jpg,halifor.jpg
3,"Phaten, Battle Alchemist",phaten.alchemical_set,alchemical_set,Alchemical Set,Brutal,Normal,5,"Equip: Whenever You Spell Play, Stack + 1\n\nStrike 1: Set Attack Damage to 4",,Game Images/Hero/phaten.jpg,Game Images/hero cards/alchemical_set.jpg,phaten.jpg
4,Machine Knight 2 - Hydra,hydra.anchor_of_the_depths,anchor_of_the_depths,Anchor of the Depths,Brutal,Reaction-Attack,8,Ferocity 5: Heal 9 Hero\nEquip: Nothing\nCharge 3: Set Toughness 1d6,,Game Images/Hero/hydra.jpg,Game Images/hero cards/anchor_of_the_depths.jpg,hydra.jpg


In [40]:
def hero_cards(hero_nickname):
    '''Return all Legendary cards that belong to a given Hero.'''
    return legends[legends['Origin'] == hero_nickname][['Name', 'Type', 'Subtype', 'Cost']]

# Change 'Akiem' to any Hero nickname
hero_cards('Machine Knight 2 - Hydra')

,Name,Type,Subtype,Cost
4,Anchor of the Depths,Brutal,Reaction-Attack,8
218,Return to the Sea,Ritual,Entropy,6


## 9. Live analytics hook

If your Flask analytics server is running, you can merge live playtest data into the static card data. If it's not running, these calls return `None` silently — no exceptions.

Requires: `python backend/analytics_server.py` running in another terminal.

In [42]:
sessions = query.live_sessions()
if sessions is None:
    print('Flask server not reachable at localhost:5050 (this is fine, skip this section)')
else:
    print(f'Found {len(sessions)} playtest sessions')
    sessions.head()

Found 0 playtest sessions


In [19]:
# Merge hero data with live pick/defeat stats
combined = query.with_live_analytics('hero')
combined.head()

,ID,Name,Passive,Passive Effect,Health,Might,Speed,Luck,Arcana,Origin,Flavor Text,Card1 Name,Card1 Type,Card1 Subtype,Card1 Cost,Card1 Effect,Card1 Flavor Text,Card2 Name,Card2 Type,Card2 Subtype,Card2 Cost,Card2 Effect,Card2 Flavor Text,Difficulty,Game Name,Alignment,Display Name,Nickname,Status,Image URL,Synergy,Hints,Icons
0,1,"Akiem, Reaper of Peace",Eyes of Blight,"Whenever You Enchant, Fortune 3\n\nWhenever You Rally, Enemy Weaken Speed 1",3,4,6,3,2,Cresia,"The Nexus's power is split across the three Relics of Humanity. Peace, Wrath...",Demise of the Blight,Ritual,Entropy,7,Deal 5 Hero\nForce Charge 2: Obliterate 1 the Target's Hand\nthe Target Ench...,"We say Blight, but there are a few types. The shades dissapate just fine. Fo...",Message for Peace,Brutal,Normal,3,Attune 2\nRally: Your Hero Boost Arcana 2,The look on her face when we saw the Message of Resolve said it all. I was n...,1,akiem,Blessed,"Akiem, Reaper of Peace",Akiem,Live,Game Images/Hero/akiem.jpg,,,
1,2,"Alverus, Devil's Finger",Devil's Favor,"Whenever You Play Normal Ritual, Blood Magic 15: Fortune 3\n\nWhenever You M...",4,4,3,3,4,Solenak,"A devil always swears its deals in Blood, and makes oaths of Hellfire. Oh Sl...",Infernal Contract,Ritual,Entropy,6,Forbid 1: Set Burn 1\nBlood Price 6: Exhume 12\n\nKill all Heroes,The powers of the Hand's Hell is limitless!,Devil's Immortality,Ritual,Entropy,3,Blood Price 3: Your Hero Set Indestructible 1\nExhume 3,Life for a price.,2,alverus,Cursed,"Alverus, Devil's Finger",Alverus,Live,Game Images/Hero/alverus.jpg,"Kalibar, Jeset, Verrida, Goliath, Lasmeer","Blood Magic, Fortune, Mercy",
2,3,"Anemos, the Insane",Joyous Madness,"Wild +1\n\nWhenever You Fortune #, Heal Fortuned Hero.",4,5,3,4,2,"Lashiq, the Unliving Sands",A man? Not quite so. \nA Devil? Not quite so. \nPositively mad with the Dark...,Enlightened Madness,Ritual,Entropy,4,Wild 6\nFortune 3\nBolster: Heal double Fortuned Hero,Things cost what they cost and do what they do...what's the cost of a little...,Warp Release: Insanity Rises,Ritual,Entropy,10,Fortune 6\nForce Blood Price Fortuned: Enemies Weaken Might Fortuned\nYou Bo...,"For centuries, The Grimoire of Chaos only had three Releases. But with the W...",4,anemos,Blessed,"Anemos, the Insane",Anemos,Live,Game Images/Hero/anemos.jpg,"Jeki, Negaknight, Nth Knight",,
3,4,"Anok'sa, the Buried Empress",Caress of the Tomb,"Whenever You Harness, Attune 2.\n\nWhenever Embalm, ME Set Toughness 1d4.",3,4,2,5,4,The Dark,,Lover's Snare,Spell,Instant,4,Harness 3: Add 2 Deny to Lover's Snare\nDeny 4\nMercy: Ally Set Toughness 1d...,Doing whatever it takes to prove a point... it has its benefits.,Embrace of the Mistress,Brutal,Normal,3,Attune 3\nUpcast 3: Obliterate 1 Your Hand\nBolster: Deal Obliterated Cost Hero,"Before the Sands began choking the Earth, my father would present fighters f...",2,anoksa,Cursed,"Anok'sa, the Buried Empress",Anok'sa,Live,Game Images/Hero/anoksa.jpg,Soul Legion Sorcerer,,
4,5,"Anya, She Who Ends Night",Waking Sky,"Whenever You Attune #, Foe Attune -1d2\n\nWhenever You Exhume #, Deal Exhume...",5,3,3,3,4,Funar'e,"For a thousand of years, the Longnight terrorized all of Funar'e. And now th...",Soul Exchange,Ritual,Normal,3,Pray: Remove 1 Tomb counter;\nRemove all Tomb counters 1 Card Your Tomb\nExh...,To save a soul lost in the Longnight means to condemn another.,Tethered Life,Spell,Normal,7,Harness 5: Remove 1 Tomb counter\nMercy: Exhume 8\n : Bolster: You ...,"Even when you reject My Gifts, I will still come through for you.",4,anya,Blessed,"Anya, She Who Ends Night",Anya,Live,Game Images/Hero/anya.jpg,,,


## 10. Your scratchpad

Everything below is yours. Ask questions, build plots, explore patterns. When a question becomes something you'll ask again, promote it to `query.py` as a function.

In [20]:
# Example: which cards have the highest average cost?
all_cards.nlargest(10, 'cost_avg')[['Name', 'Type', 'Subtype', 'Cost', 'cost_avg', 'card_source']]

,Name,Type,Subtype,Cost,cost_avg,card_source
20,Forbid,Ritual,Entropy,12,12.0,common
24,Hex of Hope,Ritual,Entropy,12,12.0,common
36,Portal to the Hells,Ritual,Entropy,12,12.0,common
42,Savage Fireball,Ritual,Entropy,12,12.0,common
43,Sever Soul,Ritual,Entropy,12,12.0,common
56,The Blade Waltz,Ritual,Entropy,12,12.0,common
66,Path of Ice,Ritual,Entropy,12,12.0,common
67,Path of Vitality,Ritual,Entropy,12,12.0,common
78,Stolen Glory,Ritual,Entropy,12,12.0,common
120,Castor Lance,Spell,Instant,12,12.0,common


In [21]:
# Example: regex search flavor text for recurring themes
import re

def search_flavor(pattern, df=heroes, col='Flavor Text'):
    if col not in df.columns:
        return pd.DataFrame()
    mask = df[col].astype(str).str.contains(pattern, case=False, regex=True, na=False)
    return df[mask][['Nickname' if 'Nickname' in df.columns else 'Name', col]]

# Change the pattern to whatever theme you want to explore
# search_flavor(r'\b(blood|curse|shadow)\b')